In [1]:
import pandas as pd
from gliner import GLiNER
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [4]:
#load dataset

df=pd.read_csv(
    "../Data/processed_healthcare_dataset.csv"
)
print(df.shape)
display(df.head())

(111722, 5)


,retrieval_text,patient_query,doctor_response,query_tokens,response_tokens
0,\n Patient Query:\n I woke up this morni...,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most...",143,135
1,\n Patient Query:\n My baby has been poo...,My baby has been pooing 56 times a day for a w...,Hi... Thank you for consulting in Chat Doctor....,74,115
2,"\n Patient Query:\n Hello, My husband is...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ...",69,179
3,\n Patient Query:\n lump under left nipp...,lump under left nipple and stomach pain male H...,HI. You have two different problems. The lump ...,65,47
4,\n Patient Query:\n I have a 5 month old...,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...,65,122


In [5]:
#load gliner model
model=GLiNER.from_pretrained(
    "urchade/gliner_medium-v2.1"
)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [6]:
labels=[
    "symptom",
    "disease",
    "medication",
    "body_part",
    "medical_condition",
    "treatment"
]

In [7]:
sample_text="""
I have cough,fever and chest pain from 2 days
"""

entities=model.predict_entities(
    sample_text,
    labels
)
print(entities)

[{'start': 8, 'end': 13, 'text': 'cough', 'label': 'medical_condition', 'score': 0.6767899990081787}, {'start': 14, 'end': 19, 'text': 'fever', 'label': 'symptom', 'score': 0.5497427582740784}, {'start': 24, 'end': 34, 'text': 'chest pain', 'label': 'symptom', 'score': 0.8090997934341431}]


In [8]:
#extract entities function

def extract_entities(text):
    try:
        if pd.isna(text):
            return []
        text=str(text)
        entities=model.predict_entities(
            text,
            labels
        )
        extracted=[]
        for entity in entities:
            extracted.append({
                "entity_text":entity["text"],
                "entity_label":entity["label"],
                "confidence":round(
                    entity["score"],4
                )
            })
        return extracted
    except Exception as e:
        print(e)
        return []

In [10]:
#test extraction function

test_query="""
I have severe cough and fever
"""
result=extract_entities(
    test_query
)

print(result)

[{'entity_text': 'fever', 'entity_label': 'symptom', 'confidence': 0.5546}]


In [14]:
all_entities=[]

sample_df=df.head(1000)

for index,row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df)
):
    try:
        text=str(
            row["retrieval_text"]
        )
        entities=extract_entities(
            text
        )
        for entity in entities:
            all_entities.append({
                "row_id":index,
                "original_text":text,
                "entity_text":entity["entity_text"],
                "entity_label":entity["entity_label"],
                "confidence":entity["confidence"]

            })
    except Exception as e:
        print(e)

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [15:08<00:00,  1.10it/s]


In [15]:
#create dataframe

entity_df=pd.DataFrame(
    all_entities
)
print(entity_df.shape)
display(entity_df.head())

(9281, 5)


,row_id,original_text,entity_text,entity_label,confidence
0,0,\n Patient Query:\n I woke up this morni...,panadol,medication,0.6307
1,0,\n Patient Query:\n I woke up this morni...,benign paroxysmal positional vertigo,medical_condition,0.7313
2,0,\n Patient Query:\n I woke up this morni...,BPPV,medical_condition,0.5871
3,0,\n Patient Query:\n I woke up this morni...,peripheral vertigo,medical_condition,0.5706
4,0,\n Patient Query:\n I woke up this morni...,dizziness,symptom,0.8374


In [16]:
#remove duplicates
entity_df=entity_df.drop_duplicates()
print(entity_df.shape)

(9276, 5)


In [17]:
#entity distribution
print(
    entity_df["entity_label"]
    .value_counts()
)

entity_label
body_part            2440
symptom              1915
medication           1656
medical_condition    1210
disease              1185
treatment             870
Name: count, dtype: int64


In [18]:
#top extracted entities
print(
    entity_df["entity_text"]
    .value_counts()
    .head(20)
)

entity_text
pain           484
treatment      107
fever          106
chest          100
symptoms        99
liver           88
antibiotics     82
neck            74
stomach         73
infection       72
heart           67
cough           63
abdomen         52
back            45
swelling        40
face            39
knee            38
diabetes        37
brain           35
cancer          32
Name: count, dtype: int64


In [22]:
#save entities
entity_df.to_csv(
    "../Data/extracted_entities.csv",
    index=False
)
print("saved successfully")

saved successfully


In [24]:
display(
    entity_df.sample(10)
)

,row_id,original_text,entity_text,entity_label,confidence
5741,616,"\n Patient Query:\n Hi,Ive been having a...",extraction,treatment,0.7989
5465,587,\n Patient Query:\n My grandson has to h...,transplantation,treatment,0.9299
3679,407,"\n Patient Query:\n Hello, about 2 month...",bruise,symptom,0.5653
915,97,"\n Patient Query:\n hi, i have just got ...",Rabies,disease,0.9754
5307,573,\n Patient Query:\n I have had a very ba...,chest,body_part,0.7903
2876,319,"\n Patient Query:\n Hi doctor,I had a sm...",treatment,treatment,0.6541
3947,435,\n Patient Query:\n My father is sufferi...,kidney,body_part,0.9739
4685,511,"\n Patient Query:\n Hi, my bf is experie...",pantoprazole,medication,0.9387
5294,572,\n Patient Query:\n Hi I am 30 yrs old a...,Lu pron Depot,medication,0.5024
3896,429,\n Patient Query:\n I have just started ...,lorazepam,medication,0.9125
